Download 100k

In [89]:
import os, zipfile, urllib.request
from pathlib import Path
import numpy as np
import pandas as pd
import random
from collections import defaultdict
from dataclasses import dataclass
from typing import List, Dict, Any, Tuple

DATA_DIR = Path("data_movielens_100k")
DATA_DIR.mkdir(parents=True, exist_ok=True)

URL = "https://files.grouplens.org/datasets/movielens/ml-100k.zip"
zip_path = DATA_DIR / "ml-100k.zip"

if not zip_path.exists():
    print("Downloading MovieLens 100K...")
    urllib.request.urlretrieve(URL, zip_path.as_posix())
else:
    print("Zip already downloaded:", zip_path)

# Unzip
with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall(DATA_DIR)

print("Extracted to:", DATA_DIR)
print("Top-level contents:", list(DATA_DIR.iterdir())[:5])

Zip already downloaded: data_movielens_100k\ml-100k.zip
Extracted to: data_movielens_100k
Top-level contents: [WindowsPath('data_movielens_100k/ml-100k'), WindowsPath('data_movielens_100k/ml-100k.zip')]


Load ratings (u.data)

In [90]:
ratings_path = DATA_DIR / "ml-100k" / "u.data"
# u.data format: user_id \t item_id \t rating \t timestamp
ratings = pd.read_csv(
    ratings_path,
    sep="\t",
    header=None,
    names=["user_id", "item_id", "rating", "timestamp"],
)

ratings.head(), ratings.shape

(   user_id  item_id  rating  timestamp
 0      196      242       3  881250949
 1      186      302       3  891717742
 2       22      377       1  878887116
 3      244       51       2  880606923
 4      166      346       1  886397596,
 (100000, 4))

Map raw IDs to contiguous indices

In [91]:
# Create contiguous indices for users and items
user_ids = ratings["user_id"].unique()
item_ids = ratings["item_id"].unique()

user2idx = {u:i for i,u in enumerate(sorted(user_ids))}
item2idx = {m:i for i,m in enumerate(sorted(item_ids))}

ratings["u"] = ratings["user_id"].map(user2idx).astype(int)
ratings["i"] = ratings["item_id"].map(item2idx).astype(int)

n_users = len(user2idx)
n_items = len(item2idx)

print("n_users:", n_users, "n_items:", n_items)
ratings[["u","i","rating"]].head()

n_users: 943 n_items: 1682


,u,i,rating
0,195,241,3
1,185,301,3
2,21,376,1
3,243,50,2
4,165,345,1


Train/test split (random 80/20 over observed ratings)

In [92]:
rng = np.random.default_rng(42)
idx = np.arange(len(ratings))
rng.shuffle(idx)

split = int(0.8 * len(idx))
train_idx = idx[:split]
test_idx  = idx[split:]

train = ratings.iloc[train_idx].reset_index(drop=True)
test  = ratings.iloc[test_idx].reset_index(drop=True)

print("Train size:", len(train), "Test size:", len(test))

# Convert to numpy arrays for fast loops
train_u = train["u"].to_numpy(np.int64)
train_i = train["i"].to_numpy(np.int64)
train_r = train["rating"].to_numpy(np.float32)

test_u = test["u"].to_numpy(np.int64)
test_i = test["i"].to_numpy(np.int64)
test_r = test["rating"].to_numpy(np.float32)

Train size: 80000 Test size: 20000


Candidate Generator

In [93]:
M = 10000
all_users = set(train['user_id'].unique())

user = random.choice(list(all_users))

def get_seen_movies(user_id, ratings):
    return set(ratings[ratings['user_id'] == user_id]['item_id'])

def generate_candidates(user, ratings):
    all_movies = set(ratings['item_id'].unique())
    seen_movies = get_seen_movies(user, ratings)

    return list(all_movies - seen_movies)

movies_unreview = generate_candidates(user, train)

def topM(ratings, user, M):
    candidates = generate_candidates(user, ratings)
    return candidates[:M]

movies_unreview = topM(train, user, M)

print(movies_unreview)

[np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18), np.int64(19), np.int64(20), np.int64(21), np.int64(22), np.int64(23), np.int64(24), np.int64(25), np.int64(26), np.int64(27), np.int64(28), np.int64(29), np.int64(30), np.int64(31), np.int64(32), np.int64(33), np.int64(34), np.int64(35), np.int64(36), np.int64(37), np.int64(38), np.int64(39), np.int64(40), np.int64(41), np.int64(42), np.int64(43), np.int64(44), np.int64(45), np.int64(46), np.int64(47), np.int64(48), np.int64(49), np.int64(50), np.int64(51), np.int64(52), np.int64(53), np.int64(54), np.int64(55), np.int64(56), np.int64(57), np.int64(58), np.int64(59), np.int64(60), np.int64(61), np.int64(62), np.int64(63), np.int64(64), np.int64(65), np.int64(66), np.int64(67), np.int64(68), np.int64(69), np.int64(70), np.int64(71), np.int64(72), 

Ranker:

. Popularity baseline:

In [94]:
def popularity_ranker(train_df):
    movie_counts = (
        train_df
        .groupby("item_id")
        .size()
        .reset_index(name="interaction_count")
        .sort_values(by="interaction_count", ascending=False)
    )
    return movie_counts

popularity_df = popularity_ranker(train)
popularity_m = popularity_ranker(train.head(M))

print(popularity_m)

print(popularity_df.head())

      item_id  interaction_count
249       258                 62
95        100                 59
277       288                 59
47         50                 58
174       181                 57
...       ...                ...
1196     1487                  1
1197     1490                  1
1198     1496                  1
1199     1501                  1
1200     1509                  1

[1227 rows x 2 columns]
     item_id  interaction_count
49        50                453
257      258                419
99       100                419
180      181                405
285      286                384


MF initialization:

In [95]:
d = 20
lam = 0.05
rng = np.random.default_rng(0)
P = 0.1 * rng.standard_normal((n_users, d)).astype(np.float32)
Q = 0.1 * rng.standard_normal((n_items, d)).astype(np.float32)
bu = np.zeros(n_users, dtype=np.float32)
bi = np.zeros(n_items, dtype=np.float32)

print("Initialized P,Q,bu,bi")

def rmse(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=np.float32)
    y_pred = np.asarray(y_pred, dtype=np.float32)
    return float(np.sqrt(np.mean((y_true - y_pred)**2)))


def predict(u, i, mu, bu, bi, P, Q):
    return mu + bu[u] + bi[i] + float(P[u] @ Q[i])

def batch_predict(u_arr, i_arr, mu, bu, bi, P, Q):
    out = np.empty(len(u_arr), dtype=np.float32)
    for k,(u,i) in enumerate(zip(u_arr, i_arr)):
        out[k] = mu + bu[u] + bi[i] + float(P[u] @ Q[i])
    return out


Initialized P,Q,bu,bi


SGD

In [96]:
eta = 0.01
epochs = 10
sgd_test_rmse = [ ]
mu = train_r.mean()
for ep in range(1, epochs+1):
    # Shuffle training examples
    order = np.arange(len(train_u))
    rng.shuffle(order)

    for idx in order:
        u = train_u[idx]
        i = train_i[idx]
        r = train_r[idx]

        # prediction + error
        rhat = mu + bu[u] + bi[i] + float(P[u] @ Q[i])
        err = r - rhat

        # Corrected SGD updates
        bu[u] = bu[u] + eta * (err - lam * bu[u])
        bi[i] = bi[i] + eta * (err - lam * bi[i])
        P[u] = P[u] + eta * (err * Q[i] - lam * P[u])
        Q[i] = Q[i] + eta * (err * P[u] - lam * Q[i])

    # Track RMSE each epoch
    tr_rmse = rmse(train_r, batch_predict(train_u, train_i, mu, bu, bi, P, Q))
    te_rmse = rmse(test_r, batch_predict(test_u, test_i, mu, bu, bi, P, Q))
    sgd_test_rmse.append(te_rmse)
    print(f"Epoch {ep:02d} | train RMSE={tr_rmse:.4f} | test RMSE={te_rmse:.4f}")

def rank_sgd(u_raw, items_raw, mu, bu, bi, P, Q, user2idx, item2idx):
    if u_raw not in user2idx:
        return []

    u = user2idx[u_raw]
    scores = []

    for i_raw in items_raw:
        if i_raw not in item2idx:
            continue

        i = item2idx[i_raw]
        score = mu + bu[u] + bi[i] + P[u] @ Q[i]
        scores.append((i_raw, score))

    return [i for i, _ in sorted(scores, key=lambda x: -x[1])]

movies_sgd = rank_sgd(user, ratings['item_id'].unique(), mu, bu, bi, P, Q, user2idx, item2idx)
print(movies_sgd)

[np.int64(1430), np.int64(1016), np.int64(1566), np.int64(577), np.int64(278), np.int64(249), np.int64(1248), np.int64(1561), np.int64(259), np.int64(113), np.int64(118), np.int64(468), np.int64(766), np.int64(1001), np.int64(1324), np.int64(507), np.int64(1274), np.int64(955), np.int64(73), np.int64(1174), np.int64(1056), np.int64(1470), np.int64(572), np.int64(341), np.int64(418), np.int64(870), np.int64(915), np.int64(534), np.int64(409), np.int64(304), np.int64(75), np.int64(1608), np.int64(364), np.int64(1026), np.int64(542), np.int64(511), np.int64(1017), np.int64(392), np.int64(698), np.int64(1054), np.int64(439), np.int64(1080), np.int64(57), np.int64(20), np.int64(1089), np.int64(1390), np.int64(899), np.int64(702), np.int64(1481), np.int64(1513), np.int64(362), np.int64(1257), np.int64(316), np.int64(503), np.int64(1511), np.int64(1352), np.int64(961), np.int64(1164), np.int64(436), np.int64(287), np.int64(662), np.int64(895), np.int64(1012), np.int64(1552), np.int64(476), np

ALS


In [97]:
# For ALS we want fast access to items rated by a user and users who rated an item
user_ratings = defaultdict(list)  # u -> list of (i, r)
item_ratings = defaultdict(list)  # i -> list of (u, r)

for u,i,r in zip(train_u, train_i, train_r):
    user_ratings[int(u)].append((int(i), float(r)))
    item_ratings[int(i)].append((int(u), float(r)))

num_iterations = 10
als_test_rmse = []

# Initialize P, Q, bu, bi for ALS
P_als = 0.1 * rng.standard_normal((n_users, d)).astype(np.float32)
Q_als = 0.1 * rng.standard_normal((n_items, d)).astype(np.float32)
bu_als = np.zeros(n_users, dtype=np.float32)
bi_als = np.zeros(n_items, dtype=np.float32)

mu = train_r.mean()

# Precompute identity
I= np.eye(d + 1, dtype=np.float32)

for iteration in range(num_iterations):

    # --- Update user factors P and user biases bu ---
    # For each user u: solve for w_u = [b_u, p_u^T]^T using ridge regression
    for u_idx in range(n_users):
        user_data = user_ratings.get(u_idx, [])
        if not user_data:
            continue

        item_indices = np.array([item for item, _ in user_data], dtype=np.int64)
        ratings_for_user = np.array([rating for _, rating in user_data], dtype=np.float32)

        # Target variable for ridge regression: r'_ui = r_ui - mu - b_i
        r_prime = ratings_for_user - mu - bi_als[item_indices]

        # Feature matrix for ridge regression: X_u = [1 | Q_items]
        Q_for_user = Q_als[item_indices] # Item factors for items rated by this user
        ones_col = np.ones((len(item_indices), 1), dtype=np.float32)
        X_u = np.hstack((ones_col, Q_for_user)) # Shape: (num_ratings_by_user, d+1)

        # Solve for w_u = [b_u, p_u^T]^T using (X_u^T X_u + lambda I)^-1 X_u^T r_prime
        XtX_plus_lambdaI = X_u.T @ X_u + lam * I

        try:
            w_u = np.linalg.solve(XtX_plus_lambdaI, X_u.T @ r_prime)
            bu_als[u_idx] = w_u[0] # First element is the user bias
            P_als[u_idx] = w_u[1:]  # Remaining elements are the user factors
        except np.linalg.LinAlgError:
            print(f"LinAlgError occurred for user {u_idx}. Skipping update in iteration {iteration+1}.")
            continue


    # --- Update item factors Q and item biases bi ---
    # For each item i: solve for w_i = [b_i, q_i^T]^T using ridge regression
    for i_idx in range(n_items):
        item_data = item_ratings.get(i_idx, [])
        if not item_data:
            continue

        user_indices = np.array([user for user, _ in item_data], dtype=np.int64)
        ratings_for_item = np.array([rating for _, rating in item_data], dtype=np.float32)

        # Target variable for ridge regression: r''_ui = r_ui - mu - b_u
        r_double_prime = ratings_for_item - mu - bu_als[user_indices]

        # Feature matrix for ridge regression: X_i = [1 | P_users]
        P_for_item = P_als[user_indices] # User factors for users who rated this item
        ones_col = np.ones((len(user_indices), 1), dtype=np.float32)
        X_i = np.hstack((ones_col, P_for_item)) # Shape: (num_ratings_for_item, d+1)

        # Solve for w_i = [b_i, q_i^T]^T using (X_i^T X_i + lambda I)^-1 X_i^T r_double_prime
        XtX_plus_lambdaI = X_i.T @ X_i + lam * I

        try:
            w_i = np.linalg.solve(XtX_plus_lambdaI, X_i.T @ r_double_prime)
            bi_als[i_idx] = w_i[0] # First element is the item bias
            Q_als[i_idx] = w_i[1:]  # Remaining elements are the item factors
        except np.linalg.LinAlgError:
            print(f"LinAlgError occurred for item {i_idx}. Skipping update in iteration {iteration+1}.")
            continue

    # Evaluate RMSE after each iteration using the updated _als parameters
    tr_rmse = rmse(train_r, batch_predict(train_u, train_i, mu, bu_als, bi_als, P_als, Q_als))
    te_rmse = rmse(test_r, batch_predict(test_u, test_i, mu, bu_als, bi_als, P_als, Q_als))
    als_test_rmse.append(te_rmse)
    print(f"ALS iter {iteration+1:02d} | train RMSE={tr_rmse:.4f} | test RMSE={te_rmse:.4f}")

# Define a ranking function for ALS
def rank_als(u_raw, items_raw, mu, bu, bi, P, Q, user2idx, item2idx):
    """
    Ranks items for a given user using the trained ALS model.

    Args:
        u_raw (int): The raw user ID.
        items_raw (list): A list of raw item IDs to rank.
        mu (float): Global mean rating.
        bu (np.ndarray): User bias vector.
        bi (np.ndarray): Item bias vector.
        P (np.ndarray): User factor matrix.
        Q (np.ndarray): Item factor matrix.
        user2idx (dict): Mapping from raw user ID to contiguous index.
        item2idx (dict): Mapping from raw item ID to contiguous index.

    Returns:
        list: A list of raw item IDs, sorted by predicted score in descending order.
    """
    if u_raw not in user2idx:
        return []

    u = user2idx[u_raw]
    scores = []

    for i_raw in items_raw:
        # Skip items not found in our item index mapping
        if i_raw not in item2idx:
            continue

        i = item2idx[i_raw]
        predicted_score = mu + bu[u] + bi[i] + P[u] @ Q[i]
        scores.append((i_raw, predicted_score))

    # Sort items by predicted score in descending order
    return [item for item, _ in sorted(scores, key=lambda x: x[1], reverse=True)]

# Test the ALS ranker with the previously chosen random user
# 'user' and 'movies_unreview' were generated in an earlier cell and are available in kernel state.
print(f"\nRanking with ALS for user {user} (raw ID).")
# movies_unreview contains candidates for this user, limited by M (10000)
movies_als = rank_als(user, movies_unreview, mu, bu_als, bi_als, P_als, Q_als, user2idx, item2idx)
K = 15
print(f"Top {K} recommended movies for user {user} using ALS: {[int(x) for x in movies_als[:K]]}")

ALS iter 01 | train RMSE=0.6907 | test RMSE=1.3422
ALS iter 02 | train RMSE=0.6044 | test RMSE=1.3641
ALS iter 03 | train RMSE=0.5664 | test RMSE=1.3926
ALS iter 04 | train RMSE=0.5436 | test RMSE=1.4130
ALS iter 05 | train RMSE=0.5282 | test RMSE=1.4324
ALS iter 06 | train RMSE=0.5172 | test RMSE=1.4496
ALS iter 07 | train RMSE=0.5090 | test RMSE=1.4652
ALS iter 08 | train RMSE=0.5026 | test RMSE=1.4778
ALS iter 09 | train RMSE=0.4973 | test RMSE=1.4885
ALS iter 10 | train RMSE=0.4930 | test RMSE=1.4980

Ranking with ALS for user 220 (raw ID).
Top 15 recommended movies for user 220 using ALS: [78, 664, 567, 1157, 904, 752, 972, 694, 460, 1007, 778, 1021, 362, 1059, 789]


## MMR Implementation

Since, sometimes, we shouldn't reccommend the top 10 movies just because they are good, we need a little diversity in the midst of our final list instead of simply providing 10 Marvel movies or 10 Star wars movies/99% copies like the other methods would do.

In [98]:
import numpy as np
import pandas as pd

item_columns = [
    'movie_id', 'title', 'release_date', 'video_release_date', 'imdb_url',
    # --- These are the 19 binary genre columns ---
    'unknown', 'Action', 'Adventure', 'Animation', 'Children', 'Comedy',
    'Crime', 'Documentary', 'Drama', 'Fantasy', 'Film-Noir', 'Horror',
    'Musical', 'Mystery', 'Romance', 'Sci-Fi', 'Thriller', 'War', 'Western'
]

item_path = DATA_DIR / "ml-100k" / "u.item"

movies = pd.read_csv(
    item_path,
    sep='|',
    names=item_columns,
    encoding='latin-1'
)

# Normalization of every movie's vector

item_genres_normalized = {}
genre_cols = item_columns[5:]

for _,row in movies.iterrows():
    m_id = row['movie_id']

    genre_vec = row[genre_cols].to_numpy(dtype=float)

    magnitude = np.linalg.norm(genre_vec)

    if magnitude > 0: item_genres_normalized[m_id] = genre_vec/magnitude
    else: item_genres_normalized[m_id] = genre_vec #Fallback for edge cases (all zeros)

print(f"Loaded {len(item_genres_normalized)} normalized genre vectors")

def cosine_sim(a: np.ndarray, b: np.ndarray) -> float:
    # Since vectors are already normalized, cosine similarity is simply given by the dot product
    return float(np.dot(a,b))

def mmr_rerank(candidate_items, base_scores, item_features, K=10, alpha=0.4):
    """
    Reranks candidates using Maximal Marginal Relevance.
    
    :param candidate_items: List of candidate movie IDs (e.g., top M=80 from MF)
    :param base_scores: Dictionary mapping {movie_id: predicted_MF_score}
    :param item_features: Dictionary mapping {movie_id: normalized_genre_vector}
    :param K: The number of items to select for the final slate
    :param alpha: The diversity trade-off parameter
    :return: List of K selected movie IDs
    """


    # Since we're now working with scalars (ratings in range [1-5]), we have to normalize them so 
    # they don't overpower the rest of the normal equation (the cosine_sim of the genre vectors and the alpha)

    scores_array = np.array([base_scores[c] for c in candidate_items])
    min_score, max_score = scores_array.min(), scores_array.max()

    scaled_scores = {}

    for c in candidate_items:
        if max_score > min_score:
            scaled_scores[c] = (base_scores[c] - min_score) / (max_score - min_score)
        else:
            scaled_scores[c] = 0.5 # Fallback for when the ratings are literally  all the same

    # Now the normal mmr can be done

    selected = []
    remaining = list(candidate_items)

    for _ in range(min(K,len(candidate_items))):
        best_item = None
        best_mmr_score = -float('inf')
        
        for c in remaining:
            relevance = scaled_scores[c]

            if len(selected) == 0:
                diversity_penality = 0.0 
            else:
                c_vec = item_features[c]
                similarities = [cosine_sim(c_vec,item_features[c]) for c in selected]
                diversity_penality = max(similarities)

            mmr_score = (1-alpha) * relevance - alpha * diversity_penality

            if mmr_score > best_mmr_score:
                best_mmr_score = best_mmr_score
                best_item = c

        selected.append(best_item)
        remaining.remove(best_item)

    return selected


Loaded 1682 normalized genre vectors


## Now we can test it all

Note: Since _*rank_als*_ and _*rank_sgd*_ are returning a sorted list of movie IDs but don't provide the predicted rating score, those needed to be recalculated

In [99]:



M = 80      # Number of candidates to consider
K = 10      # Final slate size
alpha = 0.4 # Trade-off: 0.1 (Relevance focus) to 0.7 (Diversity focus)

# 2. Grab the top M candidates from ALS Base Ranker
candidate_items = movies_als[:M]

# 3. Re-calculate the actual scalar scores for those M candidates 
base_scores = {}
u_idx = user2idx[user]

for i_raw in candidate_items:
    i_idx = item2idx[i_raw]
    # Reconstruct the prediction formula exactly as it is in your ALS code:
    predicted_rating = mu + bu_als[u_idx] + bi_als[i_idx] + P_als[u_idx] @ Q_als[i_idx]
    base_scores[i_raw] = float(predicted_rating)


final_slate = mmr_rerank(candidate_items, base_scores, item_genres_normalized, K=K, alpha=alpha)

print(f"--- Final Top {K} Slate for User {user} (alpha={alpha}) ---")
for rank, m_id in enumerate(final_slate, 1):
    # Lookup the string title in the 'movies' DataFrame we created earlier
    title = movies.loc[movies['movie_id'] == m_id, 'title'].values[0]
    score = base_scores[m_id]
    print(f"{rank}. {title} | (ALS Base Score: {score:.2f})")



--- Final Top 10 Slate for User 220 (alpha=0.4) ---
1. Remains of the Day, The (1993) | (ALS Base Score: 5.61)
2. Speechless (1994) | (ALS Base Score: 5.62)
3. Exotica (1994) | (ALS Base Score: 5.62)
4. Corrina, Corrina (1994) | (ALS Base Score: 5.62)
5. Die xue shuang xiong (Killer, The) (1989) | (ALS Base Score: 5.67)
6. Anaconda (1997) | (ALS Base Score: 5.67)
7. Patton (1970) | (ALS Base Score: 5.67)
8. Rudy (1993) | (ALS Base Score: 5.68)
9. Mouse Hunt (1997) | (ALS Base Score: 5.71)
10. Ben-Hur (1959) | (ALS Base Score: 5.72)


In [ ]:


M = 80      # Number of candidates to consider
K = 10      # Final slate size
alpha = 0.4 # Trade-off: 0.1 (Relevance focus) to 0.7 (Diversity focus)

# 2. Grab the top M candidates from SGD Base Ranker
candidate_items_sgd = movies_sgd[:M]

# 3. Re-calculate the actual scalar scores for those M candidates 
base_scores_sgd = {}
u_idx = user2idx[user]

for i_raw in candidate_items_sgd:
    i_idx = item2idx[i_raw]
    # Use the base bu, bi, P, Q arrays initialized for SGD method
    predicted_rating_sgd = mu + bu[u_idx] + bi[i_idx] + P[u_idx] @ Q[i_idx]
    base_scores_sgd[i_raw] = float(predicted_rating_sgd)

final_slate_sgd = mmr_rerank(
    candidate_items_sgd, 
    base_scores_sgd, 
    item_genres_normalized, 
    K=K, 
    alpha=alpha
)

print(f"\n--- Final Top {K} Slate for User {user} using SGD (alpha={alpha}) ---")
for rank, m_id in enumerate(final_slate_sgd, 1):
    title = movies.loc[movies['movie_id'] == m_id, 'title'].values[0]
    score = base_scores_sgd[m_id]
    print(f"{rank}. {title} | (SGD Base Score: {score:.2f})")



--- Final Top 10 Slate for User 220 using SGD (alpha=0.4) ---
1. Geronimo: An American Legend (1993) | (SGD Base Score: 3.60)
2. Tin Drum, The (Blechtrommel, Die) (1979) | (SGD Base Score: 3.60)
3. One Night Stand (1997) | (SGD Base Score: 3.60)
4. Little Princess, The (1939) | (SGD Base Score: 3.60)
5. Army of Darkness (1993) | (SGD Base Score: 3.60)
6. Ladybird Ladybird (1994) | (SGD Base Score: 3.60)
7. Bogus (1996) | (SGD Base Score: 3.60)
8. Spy Hard (1996) | (SGD Base Score: 3.60)
9. Barb Wire (1996) | (SGD Base Score: 3.60)
10. Contempt (Mépris, Le) (1963) | (SGD Base Score: 3.60)
